# MobileNetV3Large — BatchNorm ablation (diagnostic experiment, not the official submitted result)

**Owner:** Aaqib  |  **Framework:** PyTorch  |  **Config:** `configs/experiment_config.md`

**Purpose:** the officially submitted run (`notebooks/mobilenetv3_aaqib.ipynb`) keeps every BatchNorm layer
frozen in eval mode throughout Phase 2, and gets 98.9% validation accuracy but only 75.8% test accuracy — a much
larger val/test gap than the other three models. This notebook tests one hypothesis: that frozen ImageNet
BatchNorm statistics are a poor match for GTSRB's image domain (traffic-sign crops vs natural photos).

**Every group template value is identical to the official run**: seed 42, batch 32, 30 epochs/phase, patience 5
(per phase), Adam with the same two-phase LR schedule, same augmentation, same 85/15 split. The **only** change is
that BatchNorm layers inside the unfrozen blocks (`features[13:]`) are allowed to train (both running statistics
and affine parameters) in Phase 2, instead of staying frozen. Frozen blocks (`features[:13]`) keep BatchNorm in
eval mode either way, since those weights are never updated.

This is judged only on validation loss/accuracy. It does not replace the official submitted result, and it is not
re-evaluated on the test set until both configurations are compared on validation first.

| Step | What happens |
|---|---|
| 1 | Setup, config, seed |
| 2 | Load shared data (test set stays unloaded until the very end) |
| 3 | Normalization + on-the-fly GPU augmentation (train only) |
| 4 | Model: pretrained backbone + new 43-class head |
| 5 | Phase 1: backbone frozen, LR 1e-3 |
| 6 | Phase 2: upper blocks unfrozen, LR 1e-5 |
| 7 | Learning curves |
| 8 | Final test evaluation (once) |
| 9 | Efficiency numbers, save outputs to `results/` |

**To run:** select the `venv` Python kernel (top right in VS Code), then Run All.
Needs `pip install ipykernel scikit-learn matplotlib` in the venv if missing.

## 1. Setup, configuration and seed

In [ ]:
import json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)

In [ ]:
# ---- Experiment configuration (mirrors configs/experiment_config.md) ----
CFG = dict(
    model="MobileNetV3Large",
    seed=42,
    num_classes=43,
    image_size=64,
    batch_size=32,
    max_epochs_per_phase=30,
    patience=5,                 # early stopping on validation loss, within each phase
    phase1_lr=1e-3,             # Phase 1: frozen backbone, train head only
    phase2_lr=1e-5,             # Phase 2: fine-tune upper layers
    unfreeze_from_block=13,     # Phase 2 unfreezes model.features[13:] (last 3 blocks + final 1x1 conv)
    aug=dict(rotation_deg=10, shift=0.10, zoom=0.10, brightness=0.20),   # no flips
    imagenet_mean=[0.485, 0.456, 0.406],
    imagenet_std=[0.229, 0.224, 0.225],
)

# ---- Paths (works whether the notebook is run from the repo root or from notebooks/) ----
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "processed" / "gtsrb_data.npz"
RESULTS_DIR = ROOT / "results" / "mobilenetv3_bn_ablation"    # separate from the official results/ files
MODELS_DIR = ROOT / "models"            # *.pt files are gitignored
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
CKPT_PATH = MODELS_DIR / "mobilenetv3_bn_ablation_best.pt"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU ONLY (training will be very slow)")
print("repo root:", ROOT)

## 2. Load the shared data

Only train and validation are loaded here. The test arrays are read once, in section 8.
Everything is kept on the GPU (about 2 GB), so batches need no CPU-to-GPU copy.

In [ ]:
assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Get gtsrb_data.npz from the group and put it in data/processed/"
data = np.load(DATA_PATH)          # lazy: arrays are read only when accessed


def to_tensor(x):                  # (N, H, W, 3) float32 in [0, 1]  ->  (N, 3, H, W)
    return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous()


X_train = to_tensor(data["X_train"]).to(device)
y_train = torch.from_numpy(data["y_train"]).long().to(device)
X_val = to_tensor(data["X_val"]).to(device)
y_val = torch.from_numpy(data["y_val"]).long().to(device)

# Sanity checks on the shared split
assert X_train.shape[1:] == (3, CFG["image_size"], CFG["image_size"])
assert 0.0 <= X_train.min().item() and X_train.max().item() <= 1.0
assert y_train.min().item() == 0 and y_train.max().item() == CFG["num_classes"] - 1
print("train:", tuple(X_train.shape), "| val:", tuple(X_val.shape))
print("val fraction: %.3f" % (len(X_val) / (len(X_train) + len(X_val))))
print("classes in train / val:", len(y_train.unique()), "/", len(y_val.unique()))

## 3. Normalization and augmentation

* **Normalization:** ImageNet mean/std, applied identically to train, val and test.
* **Augmentation (train only):** rotation +/-10 deg, shift +/-10%, zoom +/-10%, brightness +/-20%.
  No horizontal or vertical flips (they change a sign's meaning). Done per sample on the GPU with one
  affine transform per image, so it is fast and needs no DataLoader workers on Windows.

In [ ]:
MEAN = torch.tensor(CFG["imagenet_mean"], device=device).view(1, 3, 1, 1)
STD = torch.tensor(CFG["imagenet_std"], device=device).view(1, 3, 1, 1)


def normalize(x):
    return (x - MEAN) / STD


def augment(x):
    """Per-sample random rotation / shift / zoom / brightness on a GPU batch in [0, 1]."""
    n = x.shape[0]
    a = CFG["aug"]
    r = lambda: torch.rand(n, device=x.device) * 2 - 1          # uniform in [-1, 1]
    angle = r() * a["rotation_deg"] * np.pi / 180
    zoom = 1 + r() * a["zoom"]
    tx = r() * a["shift"] * 2         # normalised coords span [-1, 1], so 10% of width = 0.2
    ty = r() * a["shift"] * 2
    cos, sin = torch.cos(angle) / zoom, torch.sin(angle) / zoom
    theta = torch.stack([torch.stack([cos, -sin, tx], dim=1),
                         torch.stack([sin, cos, ty], dim=1)], dim=1)      # (n, 2, 3)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    brightness = 1 + (torch.rand(n, 1, 1, 1, device=x.device) * 2 - 1) * a["brightness"]
    return (x * brightness).clamp(0, 1)

In [ ]:
# Visual check: top row original, bottom row augmented
sample = X_train[:8]
aug = augment(sample)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for i in range(8):
    axes[0, i].imshow(sample[i].permute(1, 2, 0).cpu()); axes[0, i].axis("off")
    axes[1, i].imshow(aug[i].permute(1, 2, 0).cpu()); axes[1, i].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("augmented", loc="left")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mobilenetv3_augmentation_check.png", dpi=150)
plt.show()

## 4. Model

Pretrained MobileNetV3Large (ImageNet). The final `Linear` layer is replaced by a new 43-way layer that outputs
raw logits (no softmax layer, because `CrossEntropyLoss` applies it internally).

* **Backbone** = `model.features` (17 blocks), **head** = `model.classifier` (Linear 960->1280, Hardswish, Dropout, new Linear 1280->43).
* **Phase 1:** backbone frozen, head trained. BatchNorm in eval mode everywhere (same as the official run).
* **Phase 2:** `features[13:]` (last 3 inverted-residual blocks plus the final 1x1 conv) is also unfrozen; earlier blocks stay frozen.
* **This ablation's only change vs the official run:** in Phase 2, BatchNorm layers inside the unfrozen blocks
  switch to **train mode** — their running statistics adapt to GTSRB and their scale/shift weights train, instead
  of staying locked to ImageNet statistics. BatchNorm in the still-frozen blocks (`features[:13]`) stays in eval
  mode either way, since those weights never change.

In [ ]:
def build_model():
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, CFG["num_classes"])
    return model


def count_params(m, trainable_only=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable_only))


def set_phase1_trainable(m):
    for p in m.parameters():
        p.requires_grad = False
    for p in m.classifier.parameters():
        p.requires_grad = True


def set_phase2_trainable(m, from_block):
    set_phase1_trainable(m)
    for block in m.features[from_block:]:
        for p in block.parameters():
            p.requires_grad = True


def set_train_mode_official(m):
    """Official run's behaviour: every BatchNorm layer stays in eval mode (ImageNet statistics frozen)."""
    m.train()
    for module in m.features.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def set_train_mode_ablation(m, unfreeze_from_block):
    """Ablation behaviour: BatchNorm in the unfrozen blocks trains too; frozen blocks stay in eval mode."""
    m.train()
    for block in m.features[:unfreeze_from_block]:
        for module in block.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()


model = build_model().to(device)
criterion = nn.CrossEntropyLoss()
print("total parameters:", f"{count_params(model):,}")
print("number of feature blocks:", len(model.features))

## 5. Training and evaluation functions

Early stopping (patience 5 on validation loss) runs inside each phase and is measured against that phase's own best loss.
The checkpoint with the lowest validation loss is saved across **both** phases, and Phase 2 starts from the best Phase 1 weights.

In [ ]:
@torch.no_grad()
def evaluate(m, X, y, batch_size=256, return_logits=False):
    m.eval()
    total_loss, correct, all_logits = 0.0, 0, []
    for i in range(0, X.shape[0], batch_size):
        xb, yb = normalize(X[i:i + batch_size]), y[i:i + batch_size]
        logits = m(xb)
        total_loss += criterion(logits, yb).item() * xb.shape[0]
        correct += (logits.argmax(1) == yb).sum().item()
        if return_logits:
            all_logits.append(logits.float().cpu())
    loss, acc = total_loss / X.shape[0], correct / X.shape[0]
    return (loss, acc, torch.cat(all_logits)) if return_logits else (loss, acc)


history = {"phase": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
best = {"val_loss": float("inf")}
train_gen = torch.Generator().manual_seed(CFG["seed"])     # controls shuffling order


def run_phase(phase, lr, bn_mode_fn):
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    n, bs, no_improve, phase_best = X_train.shape[0], CFG["batch_size"], 0, float("inf")
    for epoch in range(1, CFG["max_epochs_per_phase"] + 1):
        t0 = time.time()
        bn_mode_fn()
        perm = torch.randperm(n, generator=train_gen).to(device)
        run_loss = torch.zeros((), device=device)
        run_correct = torch.zeros((), device=device)
        seen = 0
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            if len(idx) < 2:
                continue
            xb, yb = normalize(augment(X_train[idx])), y_train[idx]
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.detach() * len(idx)
            run_correct += (logits.argmax(1) == yb).sum()
            seen += len(idx)
        train_loss, train_acc = run_loss.item() / seen, run_correct.item() / seen
        val_loss, val_acc = evaluate(model, X_val, y_val)
        dt = time.time() - t0

        for k, v in zip(history, [phase, train_loss, train_acc, val_loss, val_acc, dt]):
            history[k].append(v)

        improved = val_loss < best["val_loss"]            # new overall best: save the checkpoint
        if improved:
            best.update(val_loss=val_loss, val_acc=val_acc, phase=phase, epoch_in_phase=epoch,
                        global_epoch=len(history["val_loss"]))
            torch.save(model.state_dict(), CKPT_PATH)
        if val_loss < phase_best:                          # patience counts against this phase's own best
            phase_best, no_improve = val_loss, 0
        else:
            no_improve += 1
        print(f"[Phase {phase}] epoch {epoch:02d}/{CFG['max_epochs_per_phase']} | "
              f"train loss {train_loss:.4f} acc {train_acc:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f} | "
              f"{dt:.1f}s {'*' if improved else ''}")
        if no_improve >= CFG["patience"]:
            print(f"Early stopping in phase {phase} at epoch {epoch} (no val-loss improvement for {CFG['patience']} epochs)")
            break

## 6. Phase 1: frozen backbone, train the head (Adam, LR 1e-3)

In [ ]:
set_phase1_trainable(model)
trainable_phase1 = count_params(model, trainable_only=True)
print(f"Phase 1 trainable parameters: {trainable_phase1:,}")
run_phase(1, CFG["phase1_lr"], bn_mode_fn=lambda: set_train_mode_official(model))

## 7. Phase 2: unfreeze upper blocks, fine-tune (Adam, LR 1e-5)

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))    # start from the best Phase 1 weights
set_phase2_trainable(model, CFG["unfreeze_from_block"])
trainable_phase2 = count_params(model, trainable_only=True)
print(f"Phase 2 trainable parameters: {trainable_phase2:,}")
run_phase(2, CFG["phase2_lr"],
          bn_mode_fn=lambda: set_train_mode_ablation(model, CFG["unfreeze_from_block"]))
print("\nBest checkpoint:", best)

## 8. Learning curves

In [ ]:
epochs = list(range(1, len(history["val_loss"]) + 1))
boundary = history["phase"].index(2) + 0.5 if 2 in history["phase"] else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, key, title in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")]:
    ax.plot(epochs, history[f"train_{key}"], label="train")
    ax.plot(epochs, history[f"val_{key}"], label="validation")
    if boundary:
        ax.axvline(boundary, color="grey", linestyle="--", label="Phase 1 -> Phase 2")
    ax.axvline(best["global_epoch"], color="green", linestyle=":", label="best checkpoint")
    ax.set_xlabel("Epoch (both phases)"); ax.set_ylabel(title); ax.set_title(f"MobileNetV3Large: {title}")
    ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mobilenetv3_learning_curves.png", dpi=150)
plt.show()

## 9. Final evaluation on the official test set

The test set is loaded and used **once**, with the best saved checkpoint. No decision is made from these numbers.

In [ ]:
X_test = to_tensor(data["X_test"]).to(device)
y_test = torch.from_numpy(data["y_test"]).long().to(device)

model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
test_loss, test_acc, logits = evaluate(model, X_test, y_test, return_logits=True)

probs = torch.softmax(logits.double(), dim=1).numpy()
y_true, y_pred = y_test.cpu().numpy(), probs.argmax(1)
labels = list(range(CFG["num_classes"]))

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
prec_c, rec_c, f1_c, support_c = precision_recall_fscore_support(y_true, y_pred, labels=labels, average=None, zero_division=0)
roc_auc = roc_auc_score(y_true, probs, multi_class="ovr", average="macro", labels=labels)
cm = confusion_matrix(y_true, y_pred, labels=labels)

print(f"Test accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro precision: {macro_p:.4f} | macro recall: {macro_r:.4f} | macro F1: {macro_f1:.4f}")
print(f"Macro ROC-AUC (one-vs-rest): {roc_auc:.4f} | test loss: {test_loss:.4f}")

In [ ]:
# Confusion matrix (row-normalised so rare classes are readable)
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
plt.figure(figsize=(11, 9))
plt.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="fraction of true class")
plt.xlabel("Predicted class"); plt.ylabel("True class")
plt.title("MobileNetV3Large: confusion matrix (test set, row-normalised)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mobilenetv3_confusion_matrix.png", dpi=150)
plt.show()

## 10. Efficiency numbers and saving results

In [ ]:
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


model.eval()
with torch.no_grad():
    for i in range(20):                                   # warm-up
        model(normalize(X_test[i:i + 1]))
    sync(); t0 = time.perf_counter()
    for i in range(200):
        model(normalize(X_test[i:i + 1]))
    sync()
    latency_ms = (time.perf_counter() - t0) / 200 * 1000  # batch size 1

    sync(); t0 = time.perf_counter()
    evaluate(model, X_test, y_test)
    sync()
    throughput = len(X_test) / (time.perf_counter() - t0)  # images/second, batch 256

efficiency = {
    "total_parameters": count_params(model),
    "trainable_parameters_phase1": trainable_phase1,
    "trainable_parameters_phase2": trainable_phase2,
    "model_size_mb": CKPT_PATH.stat().st_size / 1e6,
    "training_time_total_s": sum(history["epoch_time"]),
    "training_time_per_epoch_s": float(np.mean(history["epoch_time"])),
    "epochs_run_phase1": history["phase"].count(1),
    "epochs_run_phase2": history["phase"].count(2),
    "inference_latency_ms_batch1": latency_ms,
    "inference_throughput_img_per_s": throughput,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
}
for k, v in efficiency.items():
    print(f"{k}: {v:,.3f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
results = {
    "model": CFG["model"],
    "run": "batchnorm_unfrozen_ablation",
    "note": "Diagnostic only — NOT the official submitted result. Phase 2 BatchNorm trains in the "
            "unfrozen blocks instead of staying frozen at ImageNet statistics. All template values "
            "(seed, batch size, epochs/phase, LR schedule, augmentation, split) match the official run.",
    "owner": "Aaqib",
    "test": {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "macro_roc_auc_ovr": float(roc_auc),
        "loss": float(test_loss),
    },
    "best_validation": {k: (float(v) if isinstance(v, float) else v) for k, v in best.items()},
    "per_class": {
        "precision": [float(x) for x in prec_c],
        "recall": [float(x) for x in rec_c],
        "f1": [float(x) for x in f1_c],
        "support": [int(x) for x in support_c],
    },
    "confusion_matrix": cm.tolist(),
    "efficiency": efficiency,
    "history": history,
    "config": CFG,
    "environment": {"torch": torch.__version__, "torchvision": torchvision.__version__},
}
out_path = RESULTS_DIR / "mobilenetv3_test_metrics.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print("Saved", out_path)

## 11. Comparison against the official (frozen-BatchNorm) run

Loads `results/mobilenetv3_test_metrics.json` from the official submitted notebook, if it has been run,
and prints both results side by side. Decide which configuration is better using **validation** numbers;
the test numbers are shown only for completeness, since both have now been evaluated on test.

In [ ]:
official_path = ROOT / "results" / "mobilenetv3_test_metrics.json"
if official_path.exists():
    with open(official_path) as f:
        official = json.load(f)
    rows = [
        ("Validation loss (best)", official["best_validation"]["val_loss"], best["val_loss"]),
        ("Validation accuracy (best)", official["best_validation"]["val_acc"], best["val_acc"]),
        ("Test accuracy", official["test"]["accuracy"], results["test"]["accuracy"]),
        ("Test macro F1", official["test"]["macro_f1"], results["test"]["macro_f1"]),
        ("Test macro ROC-AUC", official["test"]["macro_roc_auc_ovr"], results["test"]["macro_roc_auc_ovr"]),
    ]
    print(f"{'Metric':<28} {'Official (BN frozen)':>22} {'Ablation (BN unfrozen)':>24}")
    for name, off, abl in rows:
        print(f"{name:<28} {off:>22.4f} {abl:>24.4f}")
else:
    print(f"Official results not found at {official_path} — run notebooks/mobilenetv3_aaqib.ipynb first "
          f"to enable this comparison.")

## 12. Quick error analysis (for the report discussion)

In [ ]:
train_counts = np.bincount(y_train.cpu().numpy(), minlength=CFG["num_classes"])

print("Lowest-F1 classes (class: test F1 | train images):")
for c in np.argsort(f1_c)[:5]:
    print(f"  class {c:2d}: F1 {f1_c[c]:.3f} | {train_counts[c]} train images")

off = cm.copy()
np.fill_diagonal(off, 0)
print("\nMost frequent confusions (true -> predicted):")
for flat in np.argsort(-off.ravel())[:5]:
    t, p = np.unravel_index(flat, off.shape)
    print(f"  {t:2d} -> {p:2d}: {off[t, p]} test images")